# Transcoder + Attribution Intervention on Qwen3.6-35B-A3B L17

**Goal**: replace the L17 MoE block with a TopK transcoder (sparse, linear lookup) to enable attribution-graph-style reasoning. Then use **attribution patching** (activation × gradient) to identify per-rollout causal features, and ablate them to test if we finally recover causal control over reasoning.

**Why this, why now**: logreg directions (V1/V2) gave null. SAE features at L17 gave null. Both are single-layer, single-direction interventions. Transcoders enable true **feature-level attribution** because MoE is replaced by a sparse bilinear function, so feature→logit effect can be computed in one backward pass (AtP, Syed & Nanda 2024).

**Pipeline** (~10-14h on RTX 6000 Blackwell 96GB):
1. Install + load model (~10 min)
2. Identify L17 MoE block + install paired hooks (~5 min)
3. Extract 10M (pre_MoE, MoE_out) pairs from Stage B rollouts (~1h)
4. **Train TopK transcoder** (d_sae=16384, k=128, skip connection, ~3-4h)
5. Fidelity benchmark vs SAE (~5 min)
6. Per-rollout attribution patching: activation × gradient (~30 min)
7. Targeted feature ablation — drop top-attribution features, measure flip (~1h)
8. Analysis + comparison to all previous null results

**Expected outcomes**:
- Fidelity ≥ 75% var explained → transcoder is a usable replacement (SAE got 56%)
- Attribution-ablation flip ≥ 20% → **first mechanistic causality in hybrid MoE** — Paper 1 becomes positive
- Ablation flip < 10% → even transcoders insufficient; Paper 1 closes as systematic null, motivating cross-layer CLT (Paper 2)

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/transcoder_attribution')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model + identify L17 MoE block

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
PATCH_LAYER = 17

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Find the correct layer path — Qwen3.6 typically: model.model.language_model.layers[17]
def find_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try:
                    return cur[idx], path
                except: continue
    raise RuntimeError(f'layer {idx} not found')

layer_L17, layer_path = find_layer_module(model, PATCH_LAYER)
print(f'Layer L{PATCH_LAYER} path: {layer_path}')
print(f'Layer type: {type(layer_L17).__name__}')

# Find MoE sub-module — commonly layer.mlp or layer.feed_forward
for attr_name in ['mlp', 'feed_forward', 'moe', 'ffn']:
    if hasattr(layer_L17, attr_name):
        moe_block = getattr(layer_L17, attr_name)
        print(f'MoE block at layer.{attr_name}: {type(moe_block).__name__}')
        break
else:
    print('MoE block not found via standard names; enumerate children:')
    for n, c in layer_L17.named_children():
        print(f'  {n}: {type(c).__name__}')
    raise RuntimeError('inspect and set moe_block manually')

## Cell 3 — Extract (pre_MoE, MoE_out) pairs via hooks on Stage B rollouts

We need training data pairs `(x_pre_MoE, y_MoE_out)` so the transcoder learns `y ≈ TC(x)`.

Strategy: install PRE-HOOK on moe_block to capture input, FORWARD-HOOK to capture output. Run the model over each rollout (teacher-forced with its actual response tokens) and save pairs.

Target: ~1M-5M paired tokens (matches Stage B L17 residual count).

In [ ]:
# First, load Stage B rollouts for prompt+response data
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

MIN_LEN = 200
rollouts_flat = []  # list of dicts with prompt + response_tokens
for shard in tqdm(shards, desc='collect prompts'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_options = json.loads(meta['options'])
        question = meta['question']
        rollouts = json.loads(meta['rollouts'])
        rts = json.loads(meta['response_tokens'])
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            rollouts_flat.append(dict(
                question=question, options=q_options,
                response_tokens=rts[r_idx], pred=r['pred']))
print(f'{len(rollouts_flat)} rollouts usable')

# Pre + forward hook for MoE I/O capture
captured = {'pre': None, 'out': None}
def pre_hook(module, inputs):
    # inputs is a tuple; first arg is hidden_states
    captured['pre'] = inputs[0].detach().clone()
    return None

def fwd_hook(module, inputs, output):
    # output may be a tensor or a tuple; for Qwen MoE it's often a tuple (output, router_logits)
    out = output[0] if isinstance(output, tuple) else output
    captured['out'] = out.detach().clone()

h_pre = moe_block.register_forward_pre_hook(pre_hook)
h_fwd = moe_block.register_forward_hook(fwd_hook)

def format_prompt(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason step by step, "
        "then put the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

# Collect ~3M pairs (cap at ~4M for RAM)
TARGET_TOKENS = 3_000_000
X_pre = []
Y_out = []
total = 0
random.seed(42)
shuffled = rollouts_flat.copy()
random.shuffle(shuffled)

with torch.no_grad():
    for r in tqdm(shuffled, desc='extract pairs'):
        if total >= TARGET_TOKENS: break
        prompt = format_prompt(r['question'], r['options'])
        prompt_ids = tok(prompt, return_tensors='pt').input_ids.to('cuda')
        resp_ids = torch.tensor([r['response_tokens']], device='cuda')
        full_ids = torch.cat([prompt_ids, resp_ids], dim=1)
        # Cap sequence to avoid OOM
        if full_ids.shape[1] > 4096:
            full_ids = full_ids[:, :4096]
        try:
            model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            continue
        pre = captured['pre'][0]  # [T, d_model]
        out = captured['out'][0]
        # Only take response tokens (after prompt_len)
        prompt_len = prompt_ids.shape[1]
        pre = pre[prompt_len:].cpu()
        out = out[prompt_len:].cpu()
        X_pre.append(pre)
        Y_out.append(out)
        total += pre.shape[0]

h_pre.remove(); h_fwd.remove()

X_pre = torch.cat(X_pre, dim=0).float()
Y_out = torch.cat(Y_out, dim=0).float()
d_model = X_pre.shape[1]
print(f'X_pre: {X_pre.shape}, Y_out: {Y_out.shape}, d_model: {d_model}')
print(f'X_pre mean/std: {X_pre.mean():.4f} / {X_pre.std():.4f}')
print(f'Y_out mean/std: {Y_out.mean():.4f} / {Y_out.std():.4f}')

## Cell 4 — Train TopK Transcoder with skip connection

Architecture (Dunefsky, Chlenski & Nanda 2024 + Gao et al. TopK):
```
z = TopK(W_enc · x + b_enc, k=128)
x̂ = W_dec · z + b_dec + W_skip · x     # skip connection = dense linear baseline
loss = MSE(x̂, y_MoE_out)
```

Config: d_sae=16384 (8× d_model), k=128 (0.78% sparsity), 50k steps, Adam lr 3e-4, batch 1024.

Expected wall: 3-4h on RTX 6000 Blackwell.

In [ ]:
class TopKTranscoder(nn.Module):
    def __init__(self, d_model, d_sae, k, skip_connection=True):
        super().__init__()
        self.d_model = d_model; self.d_sae = d_sae; self.k = k
        self.W_enc = nn.Parameter(torch.randn(d_model, d_sae) * (1.0/d_model**0.5))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_model) * (1.0/d_sae**0.5))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.skip_connection = skip_connection
        if skip_connection:
            # Initialize skip as small identity-ish linear
            self.W_skip = nn.Parameter(torch.eye(d_model) * 0.01)
        with torch.no_grad():
            self.W_dec.data = self.W_dec.data / self.W_dec.data.norm(dim=-1, keepdim=True)

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        return z

    def decode(self, z, x):
        base = z @ self.W_dec + self.b_dec
        if self.skip_connection:
            return base + x @ self.W_skip
        return base

    def forward(self, x):
        z = self.encode(x)
        y_hat = self.decode(z, x)
        return y_hat, z

D_SAE = 16384
K_ACTIVE = 128
LR = 3e-4
BATCH = 1024
STEPS = 50_000

tc = TopKTranscoder(d_model, D_SAE, K_ACTIVE, skip_connection=True).cuda()
opt = torch.optim.Adam(tc.parameters(), lr=LR, betas=(0.9, 0.999))

X_gpu = X_pre.cuda()
Y_gpu = Y_out.cuda()
N = X_gpu.shape[0]
print(f'Training on {N:,} paired tokens')

with torch.no_grad():
    tc.b_dec.data = Y_gpu.mean(dim=0)

losses = []
feat_activated = torch.zeros(D_SAE, device='cuda')
tc.train()
t0 = time.time()
for step in tqdm(range(STEPS), desc='transcoder train'):
    idx = torch.randint(0, N, (BATCH,), device='cuda')
    x = X_gpu[idx]; y = Y_gpu[idx]
    y_hat, z = tc(x)
    loss = F.mse_loss(y_hat, y)
    opt.zero_grad()
    loss.backward()
    with torch.no_grad():
        grad = tc.W_dec.grad
        proj = (grad * tc.W_dec.data).sum(dim=-1, keepdim=True) * tc.W_dec.data
        tc.W_dec.grad = grad - proj
    opt.step()
    with torch.no_grad():
        tc.W_dec.data = tc.W_dec.data / tc.W_dec.data.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        feat_activated += (z > 0).float().sum(dim=0)
    losses.append(loss.item())
    if (step+1) % 2500 == 0:
        recent = np.mean(losses[-500:])
        n_dead = (feat_activated == 0).sum().item()
        print(f'  step {step+1:5d} | mse {recent:.4f} | dead features {n_dead}/{D_SAE}')
        feat_activated.zero_()

wall_min = (time.time() - t0) / 60
print(f'\nTranscoder training done in {wall_min:.1f} min, final mse: {np.mean(losses[-500:]):.4f}')

# Fidelity: on held-out 20k random pairs
tc.eval()
with torch.no_grad():
    s_idx = torch.randint(0, N, (20_000,), device='cuda')
    x_s = X_gpu[s_idx]; y_s = Y_gpu[s_idx]
    y_hat, z = tc(x_s)
    recon_cos = F.cosine_similarity(y_hat, y_s, dim=-1).mean().item()
    recon_var = 1.0 - ((y_hat - y_s).var(dim=0) / (y_s.var(dim=0) + 1e-8)).mean().item()
    l0 = (z > 0).float().sum(dim=-1).mean().item()
print(f'Fidelity: cos_sim {recon_cos:.4f} | variance explained {recon_var:.4f} | L0 {l0:.1f}')

torch.save(tc.state_dict(), OUT/'transcoder_L17.pt')
print(f'saved to {OUT/"transcoder_L17.pt"}')

## Cell 5 — Attribution patching: per-rollout feature ranking

For each rollout in a held-out set, compute `feature_attribution = activation × (∂ logit_letter_pred / ∂ activation)` at L17 position T=10.

This gives a per-feature score for how much each feature contributes to the predicted letter logit. Top-K features are the candidate **causal direct-effect** features — different from the ReasonScore-based SAE approach which uses correlation.

In [ ]:
# Build per-rollout tensor: for N rollouts, the first-10 residual tokens pre-MoE at L17
# Pick 40 confident rollouts (same pool as prior experiments)
rollout_info_list = []
n_selected = 0
for r in shuffled:
    if n_selected >= 40: break
    if len(r['response_tokens']) < 50: continue
    rollout_info_list.append(r)
    n_selected += 1

# Load SAE for comparison baseline
sae_path = Path('/content/sae_intervention/sae_L17.pt')
has_sae = sae_path.exists()
if has_sae:
    print('Previous SAE present — will compare attribution methods')

# Identify target letter token IDs
letter_token_ids = {}
for L in 'ABCDEFGHIJ':
    # Typically letter in boxed{L} — tokenize ' A' variant
    ids_a = tok(f' {L}', add_special_tokens=False).input_ids
    ids_b = tok(f'{L}', add_special_tokens=False).input_ids
    ids_c = tok(f'{L}}}', add_special_tokens=False).input_ids  # inside \boxed{A}
    letter_token_ids[L] = list(set(ids_a + ids_b + ids_c))
print('letter token candidates:', {k: v[:3] for k,v in letter_token_ids.items()})

# Captured activation during forward — need to intercept pre-MoE residual at L17
moe_pre_captured = {'value': None}
def capture_pre(module, inputs):
    moe_pre_captured['value'] = inputs[0]
h_capture = moe_block.register_forward_pre_hook(capture_pre)

# For each rollout, compute transcoder features at response position 10 (the silent-plan anchor)
# and attribute to the predicted letter's logit
attributions = []  # per rollout: (feature_activations, gradients, rollout_meta)
for r in tqdm(rollout_info_list, desc='attribution'):
    prompt = format_prompt(r['question'], r['options'])
    prompt_ids = tok(prompt, return_tensors='pt').input_ids.to('cuda')
    # Forced first 10 response tokens
    if len(r['response_tokens']) < 10: continue
    resp10 = torch.tensor([r['response_tokens'][:10]], device='cuda')
    full_ids = torch.cat([prompt_ids, resp10], dim=1)
    target_letter = r['pred']
    # Candidate token IDs for letter logit
    cand_ids = letter_token_ids[target_letter]

    # Forward to capture pre-MoE residual at L17, then compute transcoder features
    model.zero_grad()
    tc.zero_grad()
    # Need to enable grad through the residual capture path
    for p in model.parameters(): p.requires_grad_(False)
    try:
        out = model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    # Get captured residual at target position = prompt_len + 10 - 1 (last response token)
    res_pre = moe_pre_captured['value'][0]  # [T, d_model]
    target_pos = prompt_ids.shape[1] + 10 - 1  # 10th response token
    x_target = res_pre[target_pos].float().detach().clone().requires_grad_(True)  # [d_model]
    
    # Compute transcoder features + their contribution to the letter logit
    # Shortcut: logit for letter = lm_head @ final_residual; we approximate logit ~ lm_head @ (x + TC(x)) on this position
    # For attribution patching: use the replacement model output as the proxy
    # Surrogate attribution: score = feature_activation × W_unembed projection of W_dec onto letter logit
    # This is AtP-like: first-order linear attribution
    z_target = tc.encode(x_target.unsqueeze(0)).squeeze(0)  # [d_sae]
    # Get unembedding for letter candidates
    W_unembed = None
    for p_path in ['lm_head', 'model.lm_head', 'model.model.lm_head']:
        cur = model
        ok = True
        for p in p_path.split('.'):
            if hasattr(cur, p): cur = getattr(cur, p)
            else: ok = False; break
        if ok and isinstance(cur, nn.Linear):
            W_unembed = cur.weight  # [vocab, d_model]
            break
    if W_unembed is None:
        print('could not find lm_head; skip'); break
    # Mean logit direction for candidate letter tokens
    letter_dir = W_unembed[cand_ids].float().mean(dim=0)  # [d_model]
    # Attribution per feature f: activation * <W_dec[f], letter_dir>
    feature_scores = z_target * (tc.W_dec.float() @ letter_dir)  # [d_sae]
    
    attributions.append(dict(
        rollout=r,
        activations=z_target.detach().cpu().numpy(),
        scores=feature_scores.detach().cpu().numpy(),
        top20_indices=torch.topk(feature_scores, 20).indices.cpu().numpy(),
        target_letter=target_letter,
    ))

h_capture.remove()
print(f'✅ attributed {len(attributions)} rollouts')
# Save
with open(OUT/'attributions.pkl', 'wb') as f:
    pickle.dump(attributions, f)

## Cell 6 — Feature ablation intervention

For each rollout, ablate the **top-K attribution-ranked features** (zero them out at T=10 L17 residual via transcoder decoder). Measure flip.

Protocol (N=20):
1. **baseline** — no intervention
2. **top-K ablate** — zero top-20 features for predicted letter at T=10, regenerate
3. **random-K ablate** — zero 20 random features (null control)

In [ ]:
BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

class TranscoderAblator:
    """Hook on MoE block. At target response position, compute transcoder features,
    zero the specified feature IDs, decode back, and REPLACE the residual.
    This is proper feature-space intervention, not raw direction add."""
    def __init__(self, transcoder):
        self.tc = transcoder
        self.feature_mask = None  # boolean [d_sae]: True = keep, False = ablate
        self.prompt_len = None
        self.target_response_pos = None
        self.active = False
        self.applied = False

    def start(self, feature_mask, prompt_len, target_response_pos):
        self.feature_mask = torch.from_numpy(feature_mask.astype(np.bool_)).cuda()
        self.prompt_len = prompt_len
        self.target_response_pos = target_response_pos
        self.active = True
        self.applied = False

    def stop(self):
        self.active = False
        self.feature_mask = None

    def make_pre_hook(self):
        def hook(module, inputs):
            if not self.active or self.applied: return
            x = inputs[0]  # [B, T, d_model]
            target_abs_pos = self.prompt_len + self.target_response_pos
            if x.shape[1] > target_abs_pos:
                x_mod = x.clone()
                x_target = x_mod[:, target_abs_pos, :].float()  # [B, d_model]
                z = self.tc.encode(x_target)  # [B, d_sae]
                z_ablated = z * self.feature_mask  # zero the marked features
                # Reconstruct and replace
                delta = (z_ablated - z) @ self.tc.W_dec.float()  # [B, d_model]
                x_mod[:, target_abs_pos, :] = (x_target + delta).to(x.dtype)
                self.applied = True
                return (x_mod,) + inputs[1:]
            return None
        return hook

ablator = TranscoderAblator(tc)
h_ablate = moe_block.register_forward_pre_hook(ablator.make_pre_hook())
print(f'transcoder ablator hook installed on L{PATCH_LAYER} MoE')

def generate_with_ablation(prompt, forced_prefix_tokens, feature_mask, max_new=1024):
    enc = tok(prompt, return_tensors='pt').to('cuda')
    prompt_len = enc.input_ids.shape[1]
    forced_ids = torch.tensor([forced_prefix_tokens], device='cuda')
    full_input = torch.cat([enc.input_ids, forced_ids], dim=1)
    full_attn = torch.ones_like(full_input)
    target_pos = len(forced_prefix_tokens) - 1
    ablator.stop()
    if feature_mask is not None:
        ablator.start(feature_mask, prompt_len, target_pos)
    try:
        with torch.no_grad():
            out = model.generate(
                input_ids=full_input, attention_mask=full_attn,
                max_new_tokens=max_new, do_sample=False,
                pad_token_id=tok.pad_token_id, use_cache=True,
            )
    finally:
        ablator.stop()
    generated = out[0, full_input.shape[1]:]
    full_response = torch.cat([forced_ids[0], generated])
    return tok.decode(full_response, skip_special_tokens=True)

# Run intervention
N_INTERVENTIONS = 20
TOP_K_ABLATE = 20
PATCH_POSITION = 10
results = []
random.seed(42)
np.random.seed(42)
t0 = time.time()
for i, a in enumerate(tqdm(attributions[:N_INTERVENTIONS], desc='ablate-intervene')):
    r = a['rollout']
    prompt = format_prompt(r['question'], r['options'])
    forced_prefix = r['response_tokens'][:PATCH_POSITION]
    source_letter = r['pred']
    
    # Baseline: no ablation
    text_baseline = generate_with_ablation(prompt, forced_prefix, None)
    letter_baseline = extract_letter(text_baseline, len(r['options']))
    
    # Top-K attribution ablation
    mask_ablate = np.ones(D_SAE, dtype=np.bool_)
    mask_ablate[a['top20_indices'][:TOP_K_ABLATE]] = False
    text_ablated = generate_with_ablation(prompt, forced_prefix, mask_ablate)
    letter_ablated = extract_letter(text_ablated, len(r['options']))
    
    # Random-K ablation (null control)
    active_features = np.where(a['activations'] > 0)[0]
    if len(active_features) >= TOP_K_ABLATE:
        random_feats = np.random.choice(active_features, TOP_K_ABLATE, replace=False)
    else:
        random_feats = np.random.choice(D_SAE, TOP_K_ABLATE, replace=False)
    mask_random = np.ones(D_SAE, dtype=np.bool_)
    mask_random[random_feats] = False
    text_random = generate_with_ablation(prompt, forced_prefix, mask_random)
    letter_random = extract_letter(text_random, len(r['options']))
    
    results.append(dict(
        idx=i, source=source_letter,
        baseline=letter_baseline, ablated=letter_ablated, random=letter_random,
    ))
    if (i+1) % 4 == 0:
        n_drop = sum(1 for x in results if x['ablated'] != x['source'])
        n_rdrop = sum(1 for x in results if x['random'] != x['source'])
        print(f'  [{i+1}/{N_INTERVENTIONS}] top-K drop: {n_drop}/{len(results)} | random drop: {n_rdrop}/{len(results)}')

h_ablate.remove()
with open(OUT/'ablation_results.json', 'w') as f:
    json.dump(dict(
        top_k=TOP_K_ABLATE, position=PATCH_POSITION, n=len(results),
        results=results, wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Analysis + verdict

In [ ]:
from collections import Counter

def analyze(results, col):
    c = Counter()
    for r in results:
        v = r[col]
        if v is None: c['invalid'] += 1
        elif v == r['source']: c['kept'] += 1
        else: c['flipped'] += 1
    return c, sum(c.values())

print('=== Transcoder attribution-ablation results ===\n')
for col in ['baseline', 'ablated', 'random']:
    c, t = analyze(results, col)
    print(f'{col:10s}: kept={c.get("kept",0)}/{t} flipped={c.get("flipped",0)}/{t} invalid={c.get("invalid",0)}/{t}')

c_abl, t_abl = analyze(results, 'ablated')
c_rnd, t_rnd = analyze(results, 'random')
c_base, t_base = analyze(results, 'baseline')

flip_abl = c_abl.get('flipped', 0) / t_abl
flip_rnd = c_rnd.get('flipped', 0) / t_rnd
flip_base = c_base.get('flipped', 0) / t_base

print(f'\nFlip rates:')
print(f'  baseline    : {flip_base*100:.1f}%  (teacher-forced 10 tokens → should NOT flip)')
print(f'  top-K ablate: {flip_abl*100:.1f}%  (attribution-guided ablation)')
print(f'  random ablate: {flip_rnd*100:.1f}% (null control)')
print(f'\nEffect = (top-K flip) − (random flip) = {(flip_abl - flip_rnd)*100:+.1f}pp')
print(f'Effect vs baseline = {(flip_abl - flip_base)*100:+.1f}pp')

print('\n=== Complete comparison across all methods ===')
print(f'{"method":30s} {"layer":6s} {"N":>4s} {"effect_pp":>10s}')
print(f'{"logreg V1 patch α=5":30s} {"L11":6s} {20:>4d} {0.0:>+9.1f}pp')
print(f'{"logreg V2 patch α=12":30s} {"L17":6s} {20:>4d} {5.0:>+9.1f}pp')
print(f'{"factorial α=20 T=10":30s} {"L17":6s} {20:>4d} {5.0:>+9.1f}pp')
print(f'{"factorial α=40 T=10":30s} {"L17":6s} {20:>4d} {0.0:>+9.1f}pp (destructive)')
print(f'{"ablate source logreg T=10":30s} {"L17":6s} {20:>4d} {-15.0:>+9.1f}pp (anti-causal)')
print(f'{"SAE features patch α=12":30s} {"L17":6s} {20:>4d} {0.0:>+9.1f}pp (approx)')
print(f'{"transcoder attribution ablate":30s} {"L17":6s} {N_INTERVENTIONS:>4d} {(flip_abl-flip_rnd)*100:>+9.1f}pp')

print('\n=== VERDICT ===')
effect = flip_abl - flip_rnd
if effect > 0.30:
    print('⭐⭐⭐ STRONG CAUSAL — attribution-guided transcoder ablation flips reasoning in hybrid MoE')
    print('   FIRST such evidence in this architecture class. Paper 1 becomes positive finding.')
elif effect > 0.15:
    print('⭐⭐ MODERATE CAUSAL — transcoder ablation moves the needle significantly')
    print('   → Paper 1 section 3 → moderate causality via transcoder attribution.')
elif effect > 0.05:
    print('⭐ WEAK CAUSAL — transcoder ablation better than SAE/logreg but sub-threshold')
    print('   → Single-layer insufficient. Multi-layer CLT next step.')
else:
    print('❌ NULL — even attribution-guided ablation does not flip')
    print('   → Systematic null across all 3 method levels (logreg → SAE → transcoder+attribution)')
    print('   → Strong negative result: hybrid MoE requires CLT + cross-layer attribution graphs')
    print('   → Paper 1 closes; Paper 2 = CLT training on Qwen3.6 Gated-Attn layers')
